### 1. Imports & constants

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import optimize, stats
from urllib.parse import urlencode

C = 5.0          # ASSUMPTION: vans can arrive up to ~5 min before the window (data can't identify this)
BETA = 1.0       # ASSUMPTION: EBP's hour-of-day effect has the same relative size as NYC's
SHARE = 0.3      # ASSUMPTION: share of log-variance shared by nearby trips (GP signal vs noise)

# EBP performance report, Jul '25–Apr '26 (EBPAC packet, July 7, 2026)
p_eb = np.array([94.7, 4.42, 0.79, 0.06])     # on time, 1–20, 21–59, 60+ min past window
p_eb = p_eb / p_eb.sum()                      # shares sum to 99.97 -> renormalize
n_eb = p_eb * 369_241                         # ADA trips in that period
EB_EDGES = [30, 50, 89]                       # minutes from window start

# EBP runs whenever AC Transit / BART run within 3/4 mi of both ends of the trip
CORE_HOURS = np.arange(5, 24)                 # BART + regular AC Transit span (weekdays)
# 00:00–04:59: All Nighter corridors only -> low-confidence prior, not shown in the chart
EBP_HOURS = np.arange(24)                     # calibrate on all hours, weighted by volume
AFTERNOON = np.arange(12, 19)                 # for the volume sensitivity check

### 2. Load NYC Access-A-Ride pickup OTP by hour (dataset dqvw-iu8s, 2024–present)

In [2]:
url = "https://data.ny.gov/resource/dqvw-iu8s.csv"
q = {"$select": "hour_of_promise_time as h, standard, sum(total_pickups) as n, "
                "sum(pickups_within_standards) as k",
     "$group": "hour_of_promise_time, standard",
     "$limit": 1000}
df = pd.read_csv(f"{url}?{urlencode(q)}")
assert df.shape[0] == 72, df.shape

wide = df.pivot(index="h", columns="standard", values="k").sort_index()
N = df.groupby("h")["n"].first().sort_index().to_numpy(float)
k15, k20, k30 = (wide[f"{m}-minutes"].to_numpy(float) for m in (15, 20, 30))
cnt = np.stack([k15, k20 - k15, k30 - k20, N - k30], axis=1)   # NYC buckets per hour

### 3. Model helpers

In [3]:
def probs(mu, s, edges):
    """Bucket probabilities of a shifted lognormal: log(L + C) ~ N(mu, s^2)."""
    F = stats.norm.cdf((np.log(np.asarray(edges, float) + C) - mu) / s)
    return np.diff(np.concatenate([[0], F, [1]]))

def multinomial_nll(counts, p):
    return -np.sum(counts * np.log(np.clip(p, 1e-300, None)))

### 4. NYC: one lognormal per hour (D = minutes after promised time)

In [4]:
def fit_hour(c_h):
    f = lambda t: multinomial_nll(c_h, probs(t[0], np.exp(t[1]), [15, 20, 30]))
    r = optimize.minimize(f, [2.0, 0.0], method="Nelder-Mead",
                          options=dict(xatol=1e-9, fatol=1e-9))
    return r.x[0], np.exp(r.x[1])

mu_h, s_h = np.array([fit_hour(cnt[h]) for h in range(24)]).T
nyc_fit = np.array([probs(mu_h[h], s_h[h], [15, 20, 30]) for h in range(24)])
rel_err = np.max(np.abs(nyc_fit - cnt / N[:, None]) / (cnt / N[:, None]), axis=1)
print(pd.DataFrame({"OTP20 %": 100 * k20 / N, "mu_h": mu_h, "s_h": s_h,
                    "worst rel. bucket err %": 100 * rel_err}).round(3).to_string())

    OTP20 %   mu_h    s_h  worst rel. bucket err %
0    84.408  2.431  0.765                    6.625
1    87.165  2.275  0.819                    7.004
2    89.810  2.137  0.843                    5.461
3    91.152  2.073  0.830                   11.946
4    92.978  2.073  0.770                    5.223
5    94.898  2.024  0.726                    5.052
6    94.051  2.057  0.739                    5.292
7    92.012  2.118  0.774                    6.614
8    90.918  2.187  0.761                    7.680
9    92.063  2.134  0.761                    7.019
10   92.983  2.059  0.777                    7.188
11   92.901  2.014  0.811                    7.054
12   91.545  2.081  0.816                    7.854
13   89.881  2.163  0.817                    7.203
14   88.642  2.195  0.836                    7.044
15   87.528  2.237  0.840                    6.792
16   87.074  2.251  0.844                    6.621
17   86.592  2.276  0.838                    6.878
18   89.273  2.165  0.837      

### 5. Calibrate to EBP's buckets (each trip's lateness relative to its own window)

In [5]:
def calibrate(hours=EBP_HOURS, afternoon_factor=1.0):
    w = np.zeros(24); w[hours] = N[hours]                    # NYC volume as proxy for EBP volume
    w[np.intersect1d(hours, AFTERNOON)] *= afternoon_factor
    w /= w.sum()
    g = mu_h - np.sum(w * mu_h)                              # hour-of-day shape, centered
    mix = lambda m, s: sum(w[h] * probs(m + BETA * g[h], s, EB_EDGES) for h in hours)
    f = lambda t: multinomial_nll(n_eb, mix(t[0], np.exp(t[1])))
    r = optimize.minimize(f, [2.6, np.log(0.6)], method="Nelder-Mead",
                          options=dict(xatol=1e-10, fatol=1e-10))
    m, s = r.x[0], np.exp(r.x[1])
    p_late = np.array([stats.lognorm(s, loc=-C, scale=np.exp(m + BETA * g[h])).sf(30)
                       for h in range(24)])
    return dict(m=m, s=s, g=g, w=w, p_late=p_late, fit=mix(m, s))

base = calibrate()
runs = {f: calibrate(afternoon_factor=f) for f in (0.8, 1.2, 1.5)}
m, s, g, w = base["m"], base["s"], base["g"], base["w"]

print(f"m = {m:.3f}, s = {s:.3f}")
print("fitted EBP buckets %:", np.round(100 * base["fit"], 3))
print("observed EBP buckets %:", np.round(100 * p_eb, 3))
for f, r in runs.items():
    d = 100 * np.max(np.abs(r["p_late"] - base["p_late"])[CORE_HOURS])
    print(f"afternoon weight x{f}: max change in core hours {d:.2f} pp")

def prior(h):
    """Prior for boarding lateness L (min from window start) for a window starting in hour h."""
    return stats.lognorm(s, loc=-C, scale=np.exp(m + BETA * g[int(h) % 24]))

def mu0(t_hours):
    """Smooth prior mean in log space (interpolated across hours, wraps at midnight)."""
    return m + BETA * np.interp(t_hours % 24, np.arange(24), g, period=24)

m = 2.594, s = 0.588
fitted EBP buckets %: [9.473e+01 4.404e+00 8.140e-01 5.200e-02]
observed EBP buckets %: [9.4728e+01 4.4210e+00 7.9000e-01 6.0000e-02]
afternoon weight x0.8: max change in core hours 0.14 pp
afternoon weight x1.2: max change in core hours 0.12 pp
afternoon weight x1.5: max change in core hours 0.25 pp


### 6. Inference for one rider

In [6]:
sig_f2, sig_n2 = SHARE * s**2, (1 - SHARE) * s**2           # GP signal vs trip noise

def predict(window_start_h, now_h=None, post=None, n=20_000, seed=0):
    """Draws of boarding lateness (min from window start).
    post = (mean, var) of the GP latent at this rider's place/time; None -> prior only."""
    rng = np.random.default_rng(seed)
    mean, var = post if post else (mu0(window_start_h), sig_f2)
    L = np.exp(rng.normal(mean, np.sqrt(var + sig_n2), n)) - C
    if now_h is not None:                                    # still waiting -> later than that
        L = L[L > (now_h - window_start_h) * 60]
    return np.maximum(L, 0.0)                                # rider boards at window start at the earliest

def p_on_time(appt_h, window_start_h, ride_min, ride_sd=8, **kw):
    L = predict(window_start_h, **kw)
    ride = np.random.default_rng(1).normal(ride_min, ride_sd, L.size)
    return np.mean(window_start_h * 60 + L + ride <= appt_h * 60)

for ws in (12.5, 12.75, 13.0):
    print(f"2:00 appt, window {int(ws)}:{int(ws % 1 * 60):02d}, ride 35 min -> "
          f"P(on time) = {p_on_time(14.0, ws, 35):.2f}")
print("Same, 1:00 window, still waiting at 1:20 ->",
      round(p_on_time(14.0, 13.0, 35, now_h=13 + 20 / 60), 2))

2:00 appt, window 12:30, ride 35 min -> P(on time) = 0.99
2:00 appt, window 12:45, ride 35 min -> P(on time) = 0.97
2:00 appt, window 13:00, ride 35 min -> P(on time) = 0.87
Same, 1:00 window, still waiting at 1:20 -> 0.39


### 7. Interactive chart

In [7]:
from ipywidgets import interact, IntSlider                  # uv add ipywidgets

BLUE, GRAY, INK, MUTED, GRID = "#2a78d6", "#a3a29c", "#0b0b0b", "#52514e", "#e6e5e0"
x = np.linspace(-C - 5, 70, 800)
hours = np.arange(24)
all_day = sum(w[h] * prior(h).pdf(x) for h in hours)
ymax = max(prior(h).pdf(x).max() for h in CORE_HOURS) * 1.15
pl = 100 * base["p_late"]
lo = 100 * np.minimum(runs[0.8]["p_late"], runs[1.5]["p_late"])
hi = 100 * np.maximum(runs[0.8]["p_late"], runs[1.5]["p_late"])

def show(hour=17):
    d = prior(hour)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)

    # Left: prior for the selected hour vs all-day average
    a1.axvspan(0, 30, color=BLUE, alpha=0.07, lw=0)
    a1.plot(x, all_day, color=GRAY, lw=1.5, ls="--", label="All-day average")
    a1.plot(x, d.pdf(x), color=BLUE, lw=2, label=f"{hour:02d}:00")
    for q_, lab in [(0.5, "median"), (0.9, "90th")]:
        v = d.ppf(q_)
        a1.axvline(v, color=MUTED, lw=1, ls=":")
        a1.text(v + 0.8, ymax * 0.9, f"{lab}\n{v:.0f} min", color=INK, fontsize=9)
    a1.set(xlim=(x[0], x[-1]), ylim=(0, ymax),
           xlabel="Minutes from window start", ylabel="Density")
    a1.set_title(f"Pickup time prior at {hour:02d}:00   P(late) = {100*d.sf(30):.1f}%",
                 loc="left", color=INK, fontsize=11)
    a1.legend(frameon=False, loc="upper right")

    # Right: late share by hour (core service hours only)
    a2.fill_between(CORE_HOURS, lo[CORE_HOURS], hi[CORE_HOURS], color=BLUE, alpha=0.18, lw=0,
                    label="Afternoon weight ×0.8–1.5")
    a2.plot(CORE_HOURS, pl[CORE_HOURS], color=BLUE, lw=2, marker="o", ms=4, label="Prior")
    a2.plot(hour, pl[hour], "o", ms=10, color=BLUE, mec="white", mew=2)
    a2.axhline(np.sum(w * pl), color=GRAY, lw=1, ls="--")    # EBP average (~5.3%)
    a2.set(xlim=(CORE_HOURS[0] - 0.5, CORE_HOURS[-1] + 0.5), xticks=CORE_HOURS[::2],
           xlabel="Hour of pickup window", ylabel="P(picked up after window) %")
    a2.set_title("Late share by hour (EBP-calibrated)", loc="left", color=INK, fontsize=11)
    a2.legend(frameon=False, loc="upper left")

    for ax in (a1, a2):
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(axis="y", color=GRID, lw=0.8); ax.set_axisbelow(True)
    plt.show()

interact(show, hour=IntSlider(value=17, min=int(CORE_HOURS[0]), max=int(CORE_HOURS[-1]), step=1, description="Hour",
                              continuous_update=False));

interactive(children=(IntSlider(value=17, continuous_update=False, description='Hour', max=23, min=5), Output(…